In [10]:
!pip install --upgrade transformers==4.45.0 huggingface_hub

  Using cached huggingface_hub-1.6.0-py3-none-any.whl.metadata (13 kB)


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
import re
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import warnings
warnings.filterwarnings("ignore")

We suggest using a single class, it will make refinement easier. 

In your implementation, feel free to update the training procedure, change model and do whatever feels right 

In [ ]:
# =============================================================================
#  BiLSTM + Dense classification head  (simplified, less overfitting)
# =============================================================================

class BiLSTMClassificationModel(nn.Module):
    """
    RoBERTa backbone -> 1-layer BiLSTM -> single dense block -> classifier.
    Deliberately kept shallow to avoid overfitting.
    """

    def __init__(
        self,
        model_name: str,
        num_labels: int = 2,
        lstm_hidden_size: int = 256,
        dropout: float = 0.15,
    ):
        super().__init__()
        self.num_labels = num_labels

        # ---------- Transformer backbone ----------
        self.transformer = RobertaModel.from_pretrained(model_name)
        self.config = self.transformer.config
        hidden_size = self.config.hidden_size                  # 768

        # ---------- 1-layer BiLSTM ----------
        self.bilstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        lstm_out_size = lstm_hidden_size * 2                   # 512

        # ---------- Single dense block ----------
        self.dense = nn.Sequential(
            nn.Linear(lstm_out_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # ---------- Classifier ----------
        self.classifier = nn.Linear(256, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        transformer_out = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        seq_out = transformer_out.last_hidden_state              # (B, L, 768)

        lstm_out, _ = self.bilstm(seq_out)                       # (B, L, 512)

        # Masked mean-pooling
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (lstm_out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = lstm_out.mean(dim=1)

        logits = self.classifier(self.dense(pooled))

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)


# =============================================================================
#  Trainer  (UniXcoder + BiLSTM only)
# =============================================================================

class CodeClassifierTrainer:
    """
    Trains UniXcoder + BiLSTM with:
      - differential LR  (backbone lower, head higher)
      - cosine scheduler + warmup
      - frequent eval for early stopping
    """

    def __init__(
        self,
        model_name: str = "microsoft/unixcoder-base",
        max_length: int = 512,
        lstm_hidden_size: int = 256,
        dropout: float = 0.15,
    ):
        self.model_name = model_name
        self.max_length = max_length
        self.lstm_hidden_size = lstm_hidden_size
        self.dropout = dropout
        self.tokenizer = None
        self.model = None
        self.num_labels = None

    @property
    def tag(self):
        return self.model_name.split("/")[-1] + "+BiLSTM"

    def load_and_prepare_data(self, sample_size=None, val_sample_size=None, random_seed=42):
        try:
            df = pd.read_parquet(
                '/kaggle/input/datasets/daniilor/semeval-2026-task13/'
                'SemEval-2026-Task13/task_a/task_a_training_set_1.parquet'
            )
            print(f"[{self.tag}] Dataset columns: {df.columns.tolist()}")
            print(f"[{self.tag}] Original dataset size: {len(df)}")

            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            df = df.dropna(subset=['code', 'label'])
            df['label'] = df['label'].astype(int)

            if sample_size is not None and sample_size < len(df):
                df = df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(
                        n=max(1, int(sample_size * len(x) / len(df))),
                        random_state=random_seed,
                    )
                ).reset_index(drop=True)
                print(f"[{self.tag}] Sampled train size: {len(df)} (seed={random_seed})")

            self.num_labels = df['label'].nunique()
            print(f"[{self.tag}] Number of unique labels: {self.num_labels}")
            print(f"[{self.tag}] Train label distribution:\n{df['label'].value_counts().sort_index()}")

            val_df = pd.read_parquet(
                '/kaggle/input/datasets/daniilor/semeval-2026-task13/'
                'SemEval-2026-Task13/task_a/task_a_validation_set.parquet'
            )
            val_df = val_df.dropna(subset=['code', 'label'])
            val_df['label'] = val_df['label'].astype(int)

            if val_sample_size is not None and val_sample_size < len(val_df):
                val_df = val_df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(
                        n=max(1, int(val_sample_size * len(x) / len(val_df))),
                        random_state=random_seed,
                    )
                ).reset_index(drop=True)
                print(f"[{self.tag}] Sampled val size: {len(val_df)} (seed={random_seed})")

            print(f"[{self.tag}] Val label distribution:\n{val_df['label'].value_counts().sort_index()}")
            print(f"[{self.tag}] Train samples: {len(df)}, Validation samples: {len(val_df)}")
            return df, val_df

        except Exception as e:
            print(f"[{self.tag}] Error loading dataset: {e}")
            raise

    def initialize_model_and_tokenizer(self):
        print(f"[{self.tag}] Initializing model and tokenizer ...")
        self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)
        self.model = BiLSTMClassificationModel(
            model_name=self.model_name,
            num_labels=self.num_labels,
            lstm_hidden_size=self.lstm_hidden_size,
            dropout=self.dropout,
        ).to('cuda')

        total = sum(p.numel() for p in self.model.parameters())
        train = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"[{self.tag}] {self.num_labels} labels | params {total:,} | trainable {train:,}")

    def tokenize_function(self, examples):
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

    def prepare_datasets(self, train_df, val_df):
        print(f"[{self.tag}] Preparing datasets ...")
        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset   = Dataset.from_pandas(val_df[['code', 'label']])

        train_dataset = train_dataset.map(self.tokenize_function, batched=True, remove_columns=['code'])
        val_dataset   = val_dataset.map(self.tokenize_function,   batched=True, remove_columns=['code'])

        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset   = val_dataset.rename_column('label', 'labels')
        return train_dataset, val_dataset

    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
        return {'accuracy': accuracy, 'f1': f1, 'precision': precision, 'recall': recall}

    def train(self, train_dataset, val_dataset,
              output_dir="./results", num_epochs=5,
              batch_size=16, learning_rate=2e-5):
        print(f"[{self.tag}] Starting training ...")

        # eval every ~0.5 epoch so early stopping actually works
        steps_per_epoch = max(1, len(train_dataset) // batch_size)
        eval_save_steps = max(50, steps_per_epoch // 2)
        print(f"[{self.tag}] steps/epoch={steps_per_epoch}, eval every {eval_save_steps} steps")

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            weight_decay=0.01,
            logging_dir=f'{output_dir}/logs',
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=eval_save_steps,
            save_strategy="steps",
            save_steps=eval_save_steps,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="cosine",
            warmup_ratio=0.1,
            save_total_limit=2,
            report_to=[],
            fp16=True,
        )

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )
        print(f"[{self.tag}] Start training")
        trainer.train()

        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        print(f"[{self.tag}] Training completed. Model saved to {output_dir}")
        return trainer

    def evaluate_model(self, trainer, val_dataset):
        print(f"[{self.tag}] Evaluating on validation set ...")
        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids
        print(f"[{self.tag}] Validation Classification Report:")
        print(classification_report(y_true, y_pred))
        return predictions

    def run_full_pipeline(self, output_dir=None,
                          num_epochs=5, batch_size=16, learning_rate=2e-5,
                          sample_size=None, val_sample_size=None, random_seed=42):
        if output_dir is None:
            output_dir = f"./results_{self.tag.replace('+', '_')}"
        try:
            train_df, val_df = self.load_and_prepare_data(
                sample_size=sample_size,
                val_sample_size=val_sample_size,
                random_seed=random_seed,
            )
            self.initialize_model_and_tokenizer()
            train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)
            trainer = self.train(
                train_dataset, val_dataset,
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
            )
            self.evaluate_model(trainer, val_dataset)
            print(f"[{self.tag}] Pipeline completed successfully!")
            return trainer
        except Exception as e:
            print(f"[{self.tag}] Error in pipeline: {e}")
            raise

print("BiLSTMClassificationModel, CodeClassifierTrainer defined.")

BiLSTMClassificationModel, CodeClassifierTrainer defined.


In [ ]:
# =============================================================================
#  Test-set evaluation  (4 categories, printed immediately)
# =============================================================================

SEEN_LANGS   = {'c++', 'cpp', 'python', 'java'}
UNSEEN_LANGS = {'go', 'php', 'c#', 'csharp', 'c', 'javascript', 'js'}
SEEN_DOMAINS   = {'algorithmic'}
UNSEEN_DOMAINS = {'research', 'production'}

def _norm(v):
    return str(v).strip().lower()


@torch.no_grad()
def evaluate_on_test(trainer_obj, parquet_path, max_length=256, batch_size=32, device=None):
    """Run inference on test parquet → return DataFrame with 'prediction' column."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = trainer_obj.tokenizer
    model = trainer_obj.model
    model.to(device)
    model.eval()

    df = pd.read_parquet(parquet_path)
    df = df.dropna(subset=['code']).reset_index(drop=True)
    codes = df['code'].tolist()

    preds = []
    for i in tqdm(range(0, len(codes), batch_size), desc="Predicting"):
        batch = codes[i:i + batch_size]
        enc = tokenizer(batch, truncation=True, padding=True,
                        max_length=max_length, return_tensors="pt")
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        ).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())

    df['prediction'] = preds
    return df


def evaluate_by_category(df, tag="Model"):
    """Print classification metrics for the 4 evaluation settings + overall."""
    # detect columns
    lang_col = next((c for c in df.columns if c.lower() in ('language', 'lang', 'programming_language')), None)
    domain_col = next((c for c in df.columns if c.lower() in ('domain', 'task_type', 'category')), None)

    if 'label' not in df.columns:
        print(f"[{tag}] No 'label' column in test set — cannot evaluate.")
        return

    if lang_col is None or domain_col is None:
        print(f"[{tag}] Missing language/domain column (cols={df.columns.tolist()}). Overall only:")
        print(classification_report(df['label'], df['prediction']))
        return

    df['_l'] = df[lang_col].apply(_norm)
    df['_d'] = df[domain_col].apply(_norm)

    settings = [
        ("(i)   Seen Lang & Seen Domain",      SEEN_LANGS,   SEEN_DOMAINS),
        ("(ii)  Unseen Lang & Seen Domain",     UNSEEN_LANGS, SEEN_DOMAINS),
        ("(iii) Seen Lang & Unseen Domain",     SEEN_LANGS,   UNSEEN_DOMAINS),
        ("(iv)  Unseen Lang & Unseen Domain",   UNSEEN_LANGS, UNSEEN_DOMAINS),
    ]

    print(f"\n{'='*70}")
    print(f"  TEST RESULTS  --  {tag}")
    print(f"{'='*70}")

    for name, langs, domains in settings:
        mask = df['_l'].isin(langs) & df['_d'].isin(domains)
        sub = df[mask]
        n = len(sub)
        if n == 0:
            print(f"\n  {name}:  ** no samples **")
            continue
        y_t, y_p = sub['label'].values, sub['prediction'].values
        acc = accuracy_score(y_t, y_p)
        p, r, f1, _ = precision_recall_fscore_support(y_t, y_p, average='weighted', zero_division=0)
        print(f"\n  {name}  (n={n})")
        print(f"    Accuracy={acc:.4f}  Prec={p:.4f}  Recall={r:.4f}  F1={f1:.4f}")
        print(classification_report(y_t, y_p, zero_division=0))

    # overall
    acc = accuracy_score(df['label'], df['prediction'])
    _, _, f1, _ = precision_recall_fscore_support(df['label'], df['prediction'], average='weighted', zero_division=0)
    print(f"\n  OVERALL  (n={len(df)})  Accuracy={acc:.4f}  F1={f1:.4f}")
    print("="*70)

    df.drop(columns=['_l', '_d'], inplace=True)

print("evaluate_on_test(), evaluate_by_category() defined.")

predict_with_trainer() defined.


In [ ]:
# =============================================================================
#  Train UniXcoder+BiLSTM, then immediately evaluate on test set
# =============================================================================

def run_model(
    num_epochs=5,
    batch_size=16,
    learning_rate=2e-5,
    max_length=256,
    random_seed=42,
    test_parquet="/kaggle/input/datasets/daniilor/semeval-2026-task13/"
                 "SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet",
    output_base="/kaggle/working",
):
    """Train UniXcoder+BiLSTM on full data, then evaluate on test (4 categories)."""

    trainer_obj = CodeClassifierTrainer(
        model_name="microsoft/unixcoder-base",
        max_length=max_length,
        dropout=0.15,
    )
    tag = trainer_obj.tag
    out_dir = os.path.join(output_base, f"results_{tag.replace('+', '_')}")

    print("=" * 70)
    print(f"  MODEL: {tag}")
    print("=" * 70)

    # --- train on full dataset ---
    hf_trainer = trainer_obj.run_full_pipeline(
        output_dir=out_dir,
        num_epochs=num_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        sample_size=None,       # full training set
        val_sample_size=None,   # full validation set
        random_seed=random_seed,
    )

    # --- immediately evaluate on test set ---
    test_df = evaluate_on_test(
        trainer_obj=trainer_obj,
        parquet_path=test_parquet,
        max_length=max_length,
        batch_size=batch_size * 2,
        device="cuda",
    )
    evaluate_by_category(test_df, tag=tag)

    del hf_trainer, trainer_obj
    torch.cuda.empty_cache()

print("run_model() defined.")

run_all_models() defined.


In [ ]:
# =============================================================================
#  Train & evaluate  --  just hit Run!
# =============================================================================

run_model(
    num_epochs=5,
    batch_size=16,
    learning_rate=2e-5,
    max_length=256,
    random_seed=42,
)


  MODEL: codebert-base
[codebert-base] Dataset columns: ['code', 'generator', 'label', 'language']
[codebert-base] Original dataset size: 500000
[codebert-base] Sampled train size: 1999 (seed=42)
[codebert-base] Number of unique labels: 2
[codebert-base] Train label distribution:
label
0     953
1    1046
Name: count, dtype: int64
[codebert-base] Sampled val size: 499 (seed=42)
[codebert-base] Val label distribution:
label
0    238
1    261
Name: count, dtype: int64
[codebert-base] Train samples: 1999, Validation samples: 499
[codebert-base] Initializing model and tokenizer ...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[codebert-base] 2 labels | params 124,647,170 | trainable 124,647,170
[codebert-base] Preparing datasets ...


Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

[unixcoder-base+BiLSTM] Starting training ...
[unixcoder-base+BiLSTM] Start training


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
500,0.204500,0.112443,0.965932,0.965940,0.966018,0.965932
1000,0.089700,0.104895,0.971944,0.971938,0.971964,0.971944


[unixcoder-base+BiLSTM] Error in pipeline: Error while serializing: I/O error: No space left on device (os error 28)


SafetensorError: Error while serializing: I/O error: No space left on device (os error 28)